# Lab Session 5 — Grover's Algorithm

Implementation of Grover's algorithm gate by gate using Qiskit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

## TODO 1 — `gen_oracle(qc, state_list)`

This function marks the states in `state_list` by flipping their amplitude sign (multiplying by -1).

**Approach:**
- For each target state, we want to apply a phase of -1 to that specific basis state.
- To mark the state `|s⟩`, we use the following trick:
  1. Apply X gates on qubits where the target bit is '0' (to convert the target state to |11...1⟩).
  2. Apply a multi-controlled-Z (or multi-controlled phase π) gate: this flips the phase only when all qubits are |1⟩.
  3. Re-apply the same X gates to restore the qubits.
- We repeat this for every state in `state_list`.
- Note: Qiskit's bit ordering is reversed (qubit 0 is the rightmost bit in the string).

In [ ]:
def gen_oracle(qc, state_list):
    """
    Applies the Grover oracle to the quantum circuit qc.
    Marks each state in state_list by multiplying its amplitude by -1.

    Parameters:
    -----------
    qc         : QuantumCircuit — the circuit to apply the oracle to
    state_list : list of str   — list of binary strings representing target states
                                 e.g. ['00', '10'] for a 2-qubit system
    """
    n = qc.num_qubits

    for state in state_list:
        # Qiskit qubit ordering: qubit 0 corresponds to the RIGHTMOST character
        # e.g. state '10' → qubit 0 = '0', qubit 1 = '1'
        # We reverse the string so index i matches qubit i
        bits = state[::-1]

        # Step 1: Apply X on qubits where the target bit is '0'
        # This maps the target state |s⟩ → |11...1⟩
        for i, bit in enumerate(bits):
            if bit == '0':
                qc.x(i)

        # Step 2: Apply multi-controlled-Z (phase flip of -1 when all qubits are |1⟩)
        # We use a multi-controlled phase gate with angle π, which is equivalent to -1
        # The control qubits are [0, 1, ..., n-2], target qubit is n-1
        controls = list(range(n - 1))
        target = n - 1
        qc.h(target)
        qc.mcx(controls, target)  # multi-controlled X (Toffoli generalised)
        qc.h(target)
        # h · X · h = Z, so mcx sandwiched by H on target = multi-controlled Z

        # Step 3: Undo the X gates to restore the original basis
        for i, bit in enumerate(bits):
            if bit == '0':
                qc.x(i)

        qc.barrier()

## TODO 2 — `gen_diffusion(qc)`

Applies Grover's diffusion operator (inversion about the mean) to all qubits.

**Structure:** $H^{\otimes n} \cdot X^{\otimes n} \cdot \text{Multi-control-Z} \cdot X^{\otimes n} \cdot H^{\otimes n}$

**Approach:**
1. Apply H on all qubits → maps |+...+⟩ to |00...0⟩.
2. Apply X on all qubits → maps |00...0⟩ to |11...1⟩.
3. Apply a multi-controlled-Z on all qubits (phase flip of -1 only on |11...1⟩).
4. Apply X on all qubits again → undo step 2.
5. Apply H on all qubits → undo step 1.

The net effect is a reflection about the uniform superposition state |s⟩.

In [ ]:
def gen_diffusion(qc):
    """
    Applies Grover's diffusion operator (inversion about the mean) to the circuit.
    Acts on all qubits of qc.

    Structure: H⊗n · X⊗n · Multi-control-Z · X⊗n · H⊗n

    Parameters:
    -----------
    qc : QuantumCircuit — the circuit to apply the diffusion operator to
    """
    n = qc.num_qubits
    qubits = list(range(n))

    # Step 1: Apply H on all qubits
    qc.h(qubits)

    # Step 2: Apply X on all qubits
    qc.x(qubits)

    # Step 3: Multi-controlled-Z (phase flip only on |11...1⟩)
    # Implemented as H · MCX · H on the target qubit
    controls = list(range(n - 1))
    target = n - 1
    qc.h(target)
    qc.mcx(controls, target)
    qc.h(target)

    # Step 4: Apply X on all qubits (undo step 2)
    qc.x(qubits)

    # Step 5: Apply H on all qubits (undo step 1)
    qc.h(qubits)

    qc.barrier()

## TODO 3 — `ideal_sim(qc)`

Exécute le circuit sur un simulateur sans bruit (Aer) et retourne les counts.  
On teste ensuite la fonction avec **1 itération de Grover sur 5 qubits**.

In [ ]:
def ideal_sim(qc, shots=4096):
    """
    Executes a quantum circuit on a noiseless Aer simulator.
    Adds measurement gates on all qubits before running.

    Parameters:
    -----------
    qc    : QuantumCircuit — circuit to simulate (without measurements)
    shots : int            — number of shots (default 4096)

    Returns:
    --------
    counts : dict — measurement outcome counts {bitstring: count}
    """
    # Copy the circuit so we don't modify the original
    qc_meas = qc.copy()
    qc_meas.measure_all()

    # Use the statevector simulator for noiseless execution
    simulator = AerSimulator(method='statevector')
    result = simulator.run(qc_meas, shots=shots).result()
    counts = result.get_counts()
    return counts


# ── Test: 1 Grover iteration on 5 qubits ──────────────────────────────────────

n_qubits = 5
# We arbitrarily choose '10101' as the target state to mark
target_state = ['10101']

# Build the circuit
qc_test = QuantumCircuit(n_qubits)

# Initial superposition: apply H on all qubits
qc_test.h(range(n_qubits))
qc_test.barrier()

# One Grover iteration: oracle + diffusion
gen_oracle(qc_test, target_state)
gen_diffusion(qc_test)

# Simulate and plot results
counts = ideal_sim(qc_test, shots=4096)

# Plot probability of each output state
states = list(counts.keys())
probs  = [counts[s] / 4096 for s in states]

plt.figure(figsize=(14, 4))
plt.bar(states, probs)
plt.xticks(rotation=90, fontsize=7)
plt.xlabel("State")
plt.ylabel("Probability")
plt.title(f"1 Grover iteration — 5 qubits — target: {target_state}")
plt.tight_layout()
plt.show()

print(f"Probability of target state {target_state[0]}: "
      f"{counts.get(target_state[0], 0) / 4096:.3f}")

## TODO 4 — `plot_amplitudes(qc)`

Affiche les amplitudes du vecteur d'état via `Statevector` (sans mesure).  
On vérifie que ça fonctionne pour un nombre de qubits et une liste d'états arbitraires.

In [ ]:
def plot_amplitudes(qc):
    """
    Computes and plots the statevector amplitudes of a quantum circuit (no measurements).

    Uses Statevector from qiskit.quantum_info to extract the exact amplitudes
    without any shot noise. Only real amplitudes are plotted (after a Grover circuit
    starting from |+...+⟩ all amplitudes remain real).

    Parameters:
    -----------
    qc : QuantumCircuit — circuit without measurement gates
    """
    sv = Statevector(qc)
    amplitudes = sv.data  # complex array of length 2^n

    n = qc.num_qubits
    num_states = 2 ** n

    # Build labels in Qiskit convention (binary strings, qubit 0 = rightmost)
    labels = [format(i, f'0{n}b') for i in range(num_states)]

    real_amps = amplitudes.real  # amplitudes are real for standard Grover circuits

    plt.figure(figsize=(max(10, num_states // 4), 4))
    colors = ['red' if a < 0 else 'steelblue' for a in real_amps]
    plt.bar(labels, real_amps, color=colors)
    plt.axhline(0, color='black', linewidth=0.8)
    plt.xticks(rotation=90, fontsize=max(5, 8 - n))
    plt.xlabel("State")
    plt.ylabel("Amplitude")
    plt.title(f"Statevector amplitudes — {n} qubits")
    plt.tight_layout()
    plt.show()


# ── Verification: arbitrary states and arbitrary number of qubits ──────────────

# Test 1: 2 qubits, mark ['00', '10']
n1 = 2
states1 = ['00', '10']
qc1 = QuantumCircuit(n1)
qc1.h(range(n1))
qc1.barrier()
gen_oracle(qc1, states1)
gen_diffusion(qc1)
print(f"Test 1 — {n1} qubits, marked: {states1}")
plot_amplitudes(qc1)

# Test 2: 3 qubits, mark ['011']
n2 = 3
states2 = ['011']
qc2 = QuantumCircuit(n2)
qc2.h(range(n2))
qc2.barrier()
gen_oracle(qc2, states2)
gen_diffusion(qc2)
print(f"Test 2 — {n2} qubits, marked: {states2}")
plot_amplitudes(qc2)

# Test 3: 4 qubits, mark ['0101', '1010']
n3 = 4
states3 = ['0101', '1010']
qc3 = QuantumCircuit(n3)
qc3.h(range(n3))
qc3.barrier()
gen_oracle(qc3, states3)
gen_diffusion(qc3)
print(f"Test 3 — {n3} qubits, marked: {states3}")
plot_amplitudes(qc3)

## TODO 5 — Grover sur 4 qubits : 1 état marqué

On cherche à quel nombre d'itérations l'amplitude de l'état cible commence à **diminuer**.

Le nombre optimal d'itérations est donné par la formule :
$$k^* = \left\lfloor \frac{\pi}{4} \sqrt{\frac{N}{M}} \right\rfloor$$
avec $N = 2^n$ (nombre total d'états) et $M$ (nombre d'états marqués).

Pour 4 qubits et 1 état marqué : $k^* = \lfloor \frac{\pi}{4} \cdot 4 \rfloor = 3$ itérations.

In [ ]:
def grover_circuit(n_qubits, state_list, n_iterations):
    """
    Builds a full Grover circuit: superposition + n_iterations × (oracle + diffusion).

    Parameters:
    -----------
    n_qubits    : int       — number of qubits
    state_list  : list[str] — target states to mark
    n_iterations: int       — number of Grover iterations

    Returns:
    --------
    qc : QuantumCircuit (without measurements)
    """
    qc = QuantumCircuit(n_qubits)

    # Initial uniform superposition
    qc.h(range(n_qubits))
    qc.barrier()

    # Apply k Grover iterations
    for _ in range(n_iterations):
        gen_oracle(qc, state_list)
        gen_diffusion(qc)

    return qc


# ── TODO 5: 4 qubits, 1 marked state ─────────────────────────────────────────

n = 4
N = 2 ** n          # total number of states = 16
M = 1               # number of marked states
target = ['1011']   # arbitrarily chosen target state

# Theoretical optimal number of iterations
k_opt = int(np.floor((np.pi / 4) * np.sqrt(N / M)))
print(f"Theoretical optimal number of iterations: k* = {k_opt}")

# Track the amplitude of the target state across iterations
max_iter = 10
target_amplitudes = []

for k in range(1, max_iter + 1):
    qc_k = grover_circuit(n, target, k)
    sv = Statevector(qc_k)

    # Find the index of the target state in Qiskit's ordering
    # Qiskit orders states as |q_{n-1}...q_1 q_0⟩, so we reverse the string
    target_idx = int(target[0][::-1], 2)
    amp = sv.data[target_idx].real
    target_amplitudes.append(amp)

# Plot amplitude vs. number of iterations
plt.figure(figsize=(8, 4))
plt.plot(range(1, max_iter + 1), target_amplitudes, marker='o', color='steelblue')
plt.axvline(k_opt, color='red', linestyle='--', label=f'Optimal k* = {k_opt}')
plt.xlabel("Number of Grover iterations")
plt.ylabel(f"Amplitude of |{target[0]}⟩")
plt.title(f"TODO 5 — Grover on {n} qubits, 1 marked state: {target}")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Find the first iteration where the amplitude starts to decrease
for k in range(1, len(target_amplitudes)):
    if target_amplitudes[k] < target_amplitudes[k - 1]:
        print(f"Amplitude starts decreasing after iteration {k} "
              f"(amplitude at k={k}: {target_amplitudes[k-1]:.4f} → {target_amplitudes[k]:.4f})")
        break

## TODO 6 — Grover sur 4 qubits : 2 états marqués

On répète l'expérience avec **M = 2** états marqués.

Nombre optimal théorique : $k^* = \lfloor \frac{\pi}{4} \sqrt{\frac{16}{2}} \rfloor = \lfloor \frac{\pi}{4} \cdot 2\sqrt{2} \rfloor = 2$ itérations.

**Conclusion attendue :** plus M est grand, moins il faut d'itérations pour converger.  
Le nombre optimal d'itérations est en $\mathcal{O}\!\left(\sqrt{N/M}\right)$ : plus il y a d'états solutions, plus Grover converge vite.

In [ ]:
# ── TODO 6: 4 qubits, 2 marked states ────────────────────────────────────────

n = 4
N = 2 ** n             # 16
M = 2                  # 2 marked states
targets_2 = ['1011', '0110']  # two arbitrarily chosen states

# Theoretical optimal number of iterations
k_opt_2 = int(np.floor((np.pi / 4) * np.sqrt(N / M)))
print(f"Theoretical optimal number of iterations (M=2): k* = {k_opt_2}")

# Track amplitude of EACH marked state across iterations
max_iter = 10
amps_state1 = []
amps_state2 = []

idx1 = int(targets_2[0][::-1], 2)
idx2 = int(targets_2[1][::-1], 2)

for k in range(1, max_iter + 1):
    qc_k = grover_circuit(n, targets_2, k)
    sv = Statevector(qc_k)
    amps_state1.append(sv.data[idx1].real)
    amps_state2.append(sv.data[idx2].real)

iterations = range(1, max_iter + 1)

plt.figure(figsize=(8, 4))
plt.plot(iterations, amps_state1, marker='o', label=f"|{targets_2[0]}⟩", color='steelblue')
plt.plot(iterations, amps_state2, marker='s', label=f"|{targets_2[1]}⟩", color='darkorange')
plt.axvline(k_opt_2, color='red', linestyle='--', label=f'Optimal k* = {k_opt_2}')
plt.xlabel("Number of Grover iterations")
plt.ylabel("Amplitude")
plt.title(f"TODO 6 — Grover on {n} qubits, 2 marked states: {targets_2}")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Report when each amplitude starts decreasing
for label, amps in zip(targets_2, [amps_state1, amps_state2]):
    for k in range(1, len(amps)):
        if amps[k] < amps[k - 1]:
            print(f"|{label}⟩ amplitude starts decreasing after iteration {k} "
                  f"({amps[k-1]:.4f} → {amps[k]:.4f})")
            break

# ── Comparison summary ────────────────────────────────────────────────────────
print()
print("=== Conclusion ===")
print(f"  M=1 marked state  → optimal k* = {int(np.floor(np.pi/4 * np.sqrt(N/1)))} iterations")
print(f"  M=2 marked states → optimal k* = {int(np.floor(np.pi/4 * np.sqrt(N/2)))} iterations")
print()
print("The optimal number of Grover iterations scales as O(sqrt(N/M)).")
print("More marked states → fewer iterations needed to amplify them.")
print("Over-iterating beyond k* causes the amplitude to oscillate back down.")

## TODO 7 — Transpilation sur topologie cycle 4 qubits

On contraint le circuit Grover (4 qubits, k* = 3 itérations) à :
- Une **topologie cycle** : q0-q1-q2-q3-q0 (seules les paires adjacentes peuvent faire un CNOT)
- Un **jeu de portes restreint** : `cx`, `rz`, `rx`, `h`

La transpilation doit décomposer les portes multi-contrôlées en ces portes élémentaires et insérer des **SWAP** si deux qubits non-connectés doivent interagir → la profondeur augmente.

In [ ]:
from qiskit import transpile
from qiskit.transpiler import CouplingMap

# ── Build the optimal Grover circuit (4 qubits, k*=3, 1 marked state) ─────────

n = 4
target = ['1011']
k_opt = 3   # from TODO 5

qc_grover = grover_circuit(n, target, k_opt)
depth_before = qc_grover.depth()
print(f"Circuit depth BEFORE transpilation : {depth_before}")
print(f"Gate counts BEFORE: {qc_grover.count_ops()}")

# ── Define the cycle coupling map: q0-q1, q1-q2, q2-q3, q3-q0 ────────────────
# Each edge must be listed in both directions for an undirected coupling
coupling_map = CouplingMap(couplinglist=[
    [0, 1], [1, 0],
    [1, 2], [2, 1],
    [2, 3], [3, 2],
    [3, 0], [0, 3]
])

# ── Define the restricted gate set ────────────────────────────────────────────
basis_gates = ['cx', 'rz', 'rx', 'h']

# ── Transpile ─────────────────────────────────────────────────────────────────
# optimization_level=3 gives the best decomposition within the constraints
qc_transpiled = transpile(
    qc_grover,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    optimization_level=3,
    seed_transpiler=42
)

depth_after = qc_transpiled.depth()
print(f"\nCircuit depth AFTER transpilation  : {depth_after}")
print(f"Gate counts AFTER : {qc_transpiled.count_ops()}")
print(f"\nDepth increase factor: x{depth_after / depth_before:.1f}")

# ── Visual comparison ─────────────────────────────────────────────────────────
print("\n--- Circuit BEFORE transpilation (first 40 ops shown) ---")
print(qc_grover.draw(output='text', fold=120))

print("\n--- Circuit AFTER transpilation (folded) ---")
print(qc_transpiled.draw(output='text', fold=120))

# ── Discussion ────────────────────────────────────────────────────────────────
print("""
=== Why does the depth increase? ===
1. Gate decomposition: multi-controlled gates (MCX, MCZ) are not native.
   They must be broken down into many CX + single-qubit gates.
2. SWAP insertion: the cycle topology only allows CNOTs between adjacent qubits.
   Non-adjacent qubits require SWAP chains to bring them together,
   each SWAP costing 3 CX gates.

=== Impact on noisy simulation ===
Each additional gate introduces errors (decoherence, gate fidelity).
A deeper circuit accumulates more errors → the probability of the target state
is degraded much more severely on a noisy device than on the ideal simulator.
""")

## TODO 8 — Simulation bruitée avec modèle d'erreur de Pauli

On crée un **modèle d'erreur de Pauli** appliqué à chaque porte du circuit transpilé.  
On augmente progressivement le taux d'erreur `p` pour observer la dégradation de l'amplitude de l'état cible.

**Modèle exponentiel :** sur un circuit de profondeur $d$ avec taux d'erreur par porte $p$,  
la fidélité décroît approximativement comme $(1-p)^d$ — une décroissance **exponentielle** en profondeur.

In [ ]:
from qiskit_aer.noise import NoiseModel, pauli_error

def build_pauli_noise_model(p_single, p_two):
    """
    Builds a Pauli error noise model.

    For single-qubit gates (h, rx, rz):
        depolarising-like Pauli error with probability p_single,
        distributed as p/3 on each of X, Y, Z.

    For two-qubit gates (cx):
        Pauli error with probability p_two on the 15 non-identity 2-qubit
        Pauli operators (equal probability p_two/15 each).

    Parameters:
    -----------
    p_single : float — total error probability for 1-qubit gates
    p_two    : float — total error probability for 2-qubit gates

    Returns:
    --------
    noise_model : NoiseModel
    """
    noise_model = NoiseModel()

    # ── Single-qubit Pauli error (X, Y, Z with equal weight) ──────────────────
    error_1q = pauli_error([
        ('X', p_single / 3),
        ('Y', p_single / 3),
        ('Z', p_single / 3),
        ('I', 1 - p_single)
    ])
    for gate in ['h', 'rx', 'rz']:
        noise_model.add_all_qubit_quantum_error(error_1q, gate)

    # ── Two-qubit Pauli error on CX ───────────────────────────────────────────
    # 15 non-identity 2-qubit Pauli operators, each with probability p_two/15
    paulis_2q = [a + b for a in 'IXYZ' for b in 'IXYZ' if a + b != 'II']
    p_each = p_two / 15
    error_2q = pauli_error([(p, p_each) for p in paulis_2q] + [('II', 1 - p_two)])
    noise_model.add_all_qubit_quantum_error(error_2q, 'cx')

    return noise_model


# ── Sweep over noise levels ────────────────────────────────────────────────────

shots = 8192
# target state index in the counts dict (Qiskit reverses bit order in keys)
target_key = target[0]  # '1011' — Qiskit measurement strings match state_list directly

# Range of noise values to test
noise_levels = np.linspace(0.0, 0.15, 20)

target_probs_noisy   = []
target_probs_noiseless = []

# Noiseless reference at each iteration count is constant — compute once
qc_meas = qc_transpiled.copy()
qc_meas.measure_all()
sim_ideal = AerSimulator(method='statevector')
counts_ideal = sim_ideal.run(qc_meas, shots=shots).result().get_counts()
prob_ideal = counts_ideal.get(target_key, 0) / shots
print(f"Noiseless probability of |{target_key}⟩ on transpiled circuit: {prob_ideal:.4f}")

for p in noise_levels:
    if p == 0.0:
        target_probs_noisy.append(prob_ideal)
        continue

    nm = build_pauli_noise_model(p_single=p, p_two=2 * p)  # 2-qubit gates noisier
    sim_noisy = AerSimulator(noise_model=nm)
    counts_noisy = sim_noisy.run(qc_meas, shots=shots).result().get_counts()
    prob = counts_noisy.get(target_key, 0) / shots
    target_probs_noisy.append(prob)

# ── Exponential fit for illustration ──────────────────────────────────────────
from scipy.optimize import curve_fit

def exp_decay(p, a, b):
    return a * np.exp(-b * p)

popt, _ = curve_fit(exp_decay, noise_levels, target_probs_noisy,
                    p0=[prob_ideal, 20], maxfev=5000)

p_fit = np.linspace(0, 0.15, 200)

# ── Plot ───────────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(noise_levels, target_probs_noisy, 'o', color='steelblue',
         label='Noisy simulation')
plt.plot(p_fit, exp_decay(p_fit, *popt), '--', color='red',
         label=f'Exponential fit: a·e^(-b·p), b={popt[1]:.1f}')
plt.axhline(prob_ideal, color='green', linestyle=':', label='Noiseless reference')
plt.xlabel("Pauli error probability per gate (p)")
plt.ylabel(f"Probability of |{target_key}⟩")
plt.title("TODO 8 — Noisy Grover: target state probability vs. error rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print("""
=== Trade-off discussion (exponential noise model) ===
- At low noise (p ≈ 0), the circuit faithfully amplifies the target state.
- As p increases, each gate contributes an independent error, and the
  overall fidelity decays roughly as (1-p)^(gate_count) ~ exp(-p * d),
  which is an exponential decay in the error rate.
- The deeper the transpiled circuit, the steeper the decay.
- Beyond a noise threshold, the target state probability drops BELOW the
  uniform background 1/N = 1/16 = 0.0625, meaning Grover is no longer
  useful: random guessing would perform just as well.
- This illustrates why error correction is essential for large Grover circuits.
""")

## TODO 9 — Grover sur 40 qubits : analyse de profondeur après transpilation

On construit un circuit Grover sur **40 qubits** avec :
- Un oracle simple = une porte de phase multi-contrôlée (`mcx` + H sur la cible)
- **20 itérations** de Grover
- Topologie **cycle 40 qubits** : q0-q1-...-q39-q0
- Même jeu de portes restreint que TODO 7 : `cx`, `rz`, `rx`, `h`

> ⚠️ On ne **simule pas** ce circuit (2^40 ≈ 10^12 états — impossible classiquement).  
> On analyse uniquement sa **profondeur** après transpilation.

In [ ]:
import time

# ── Build the 40-qubit Grover circuit ─────────────────────────────────────────

N_BIG   = 40
N_ITER  = 20

# We use a simple multi-controlled phase gate as the oracle (depth 1 oracle).
# The oracle marks |11...1⟩ (all ones) — the simplest possible target for MCX.
def build_grover_40(n_qubits, n_iterations):
    """
    Builds a Grover circuit for n_qubits with n_iterations.
    Oracle: simple multi-controlled-Z on |11...1⟩ (depth-1 oracle).
    Diffusion: standard H-X-MCZ-X-H structure.

    We do NOT simulate this circuit — only analyse its compiled depth.
    """
    qc = QuantumCircuit(n_qubits)
    qubits = list(range(n_qubits))

    # Initial uniform superposition
    qc.h(qubits)
    qc.barrier()

    for _ in range(n_iterations):
        # ── Oracle: mark |11...1⟩ via MCZ (H · MCX · H on last qubit) ────────
        # This is a depth-1 oracle (single multi-controlled gate)
        controls = list(range(n_qubits - 1))
        target   = n_qubits - 1
        qc.h(target)
        qc.mcx(controls, target)
        qc.h(target)
        qc.barrier()

        # ── Diffusion: H X MCZ X H ─────────────────────────────────────────
        qc.h(qubits)
        qc.x(qubits)
        qc.h(target)
        qc.mcx(controls, target)
        qc.h(target)
        qc.x(qubits)
        qc.h(qubits)
        qc.barrier()

    return qc

print(f"Building {N_BIG}-qubit Grover circuit with {N_ITER} iterations...")
t0 = time.time()
qc_big = build_grover_40(N_BIG, N_ITER)
print(f"  Done in {time.time()-t0:.1f}s")
print(f"  Depth before transpilation : {qc_big.depth()}")
print(f"  Gate count before          : {qc_big.count_ops()}")

# ── Define 40-qubit cycle coupling map ────────────────────────────────────────
print(f"\nBuilding {N_BIG}-qubit cycle coupling map...")
edges = []
for i in range(N_BIG):
    j = (i + 1) % N_BIG
    edges.append([i, j])
    edges.append([j, i])

coupling_map_big = CouplingMap(couplinglist=edges)

# ── Transpile ─────────────────────────────────────────────────────────────────
# optimization_level=1 for speed — level 3 would take very long on 40 qubits
print("Transpiling (this may take a minute)...")
t0 = time.time()
qc_big_transpiled = transpile(
    qc_big,
    coupling_map=coupling_map_big,
    basis_gates=['cx', 'rz', 'rx', 'h'],
    optimization_level=1,
    seed_transpiler=42
)
print(f"  Transpilation done in {time.time()-t0:.1f}s")

depth_big_before = qc_big.depth()
depth_big_after  = qc_big_transpiled.depth()
ops_after        = qc_big_transpiled.count_ops()

print(f"\n{'='*55}")
print(f"  Depth BEFORE transpilation : {depth_big_before}")
print(f"  Depth AFTER  transpilation : {depth_big_after}")
print(f"  Increase factor            : x{depth_big_after / depth_big_before:.1f}")
print(f"  Gate counts AFTER          : {ops_after}")
print(f"{'='*55}")

# ── Conclusion ────────────────────────────────────────────────────────────────
print(f"""
=== Analysis ===

Circuit depth after transpilation: {depth_big_after}

Is it reasonable to execute this on a noisy quantum computer?

NO — for several reasons:

1. Depth explosion: the cycle topology forces long SWAP chains to implement
   long-range interactions in the 40-qubit MCX gate (O(n) SWAPs per MCX,
   each SWAP = 3 CX gates). With 20 iterations the depth grows enormously.

2. Exponential error accumulation: assuming a realistic 2-qubit gate error
   rate of ~0.1-1%, the circuit fidelity decays as (1-p)^(gate_count).
   With thousands of gates, the fidelity approaches zero — the output is
   indistinguishable from a uniform random distribution.

3. Coherence time: current superconducting qubits have T2 ~ 100 µs and
   gate times ~ 50 ns. A circuit of depth {depth_big_after} would take far
   longer than the coherence time, causing the quantum state to fully decohere.

4. No error correction: without fault-tolerant quantum error correction
   (which requires ~1000 physical qubits per logical qubit), this circuit
   cannot be reliably executed on current NISQ hardware.

Conclusion: Grover's algorithm at this scale requires error-corrected
quantum hardware that does not yet exist.
""")